# QAT on Phi-4 Mini Instruct and lower & export quantized model to XNNPACK using ExecuTorch

### Imports

In [ ]:
import sys
!{sys.executable} -m pip install -U accelerate

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from torchao.quantization import quantize_, Int8DynamicActivationInt4WeightConfig
from torchao.quantization.qat import QATConfig
# Below is optional for quantizing embedding configs per channel
# from torchao.quantization.granularity import PerGroup, PerAxis

### Get Model

In [ ]:
torch.random.manual_seed(0)

model_path = "microsoft/Phi-4-mini-instruct"
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype="auto", device_map="mps")
tokenizer = AutoTokenizer.from_pretrained(model_path)

### prepare swap actual quantize operations (torch.nn.Linear) with fake quantize(FakeQuantizedLinear)
#### Quantizer for int8 dynamic per token activations + int4 grouped per channel weights, only for linear layers

In [ ]:
base_config = Int8DynamicActivationInt4WeightConfig(group_size=32)
quantize_(model, QATConfig(base_config, step="prepare"))

### Training Loop

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=1e-5)
loss_fn = torch.nn.CrossEntropyLoss()
vocab_size = model.config.vocab_size
for i in range(10):
    example = torch.randint(0, vocab_size, (2, 16)).to("mps")
    target = torch.randn((2, 16, vocab_size)).to("mps")
    output = model(example).logits
    loss = loss_fn(output, target)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

### Convert fake quantize to actual quantize operations (torch.nn.Linear)

In [ ]:
quantize_(model, QATConfig(base_config, step="convert"))

### Inference

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": "Can you provide ways to eat combinations of bananas and dragonfruits?"},
    {"role": "assistant", "content": "Sure! Here are some ways to eat bananas and dragonfruits together: 1. Banana and dragonfruit smoothie: Blend bananas and dragonfruits together with some milk and honey. 2. Banana and dragonfruit salad: Mix sliced bananas and dragonfruits together with some lemon juice and honey."},
    {"role": "user", "content": "What about solving an 2x + 3 = 7 equation?"},
]

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

generation_args = {
    "max_new_tokens": 500,
    "return_full_text": False,
    "temperature": 0.0,
    "do_sample": False,
}

output = pipe(messages, **generation_args)
print(output[0]['generated_text'])

### Export to ExecuTorch using export_llama

In [ ]:
# Install ExecuTorch in terminal
#git clone https://github.com/pytorch/executorch.git
#cd executorch
#./install_requirements.sh

In [ ]:
# Convert checkpoint format for ExecuTorch
!python -m executorch.examples.models.phi_4_mini.convert_weights pytorch_model.bin pytorch_model_converted.bin

# Export to PTE format with torchao optimizations preserved
import os
os.environ["PARAMS"]="executorch/examples/models/phi_4_mini/config.json"
!python -m executorch.examples.models.llama.export_llama \
    --model "phi_4_mini" \
    --checkpoint "pytorch_model_converted.bin" \
    --params "$PARAMS" \
    -kv \
    --use_sdpa_with_kv_cache \
    -X \
    --metadata '{"get_bos_id":199999, "get_eos_ids":[200020,199999]}' \
    --max_seq_length 128 \
    --max_context_length 128 \
    --output_name="phi4-mini-8da4w.pte"